In [35]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
cd drive/MyDrive/house_price_prediction/

[Errno 2] No such file or directory: 'drive/MyDrive/house_price_prediction/'
/content/drive/MyDrive/house_price_prediction


In [37]:
ls -l

total 1757
-rw------- 1 root root  302183 Jul 10 02:33 house.csv
-rw------- 1 root root   57444 Jul 13 03:51 house_price_model.ipynb
-rw------- 1 root root 1437367 Jul 13 03:40 house_price_model.pkl
-rw------- 1 root root     183 Jul 10 07:06 readme.md.gdoc


In [38]:
import pandas as pd

df = pd.read_csv("house.csv")
df.head()

,House_ID,Area_sqft,Bedrooms,Bathrooms,Floors,House_Age,Garage,Parking,Location,Furnished,Garden,Condition,Distance_to_City_km,Price
0,H00001,1028,3,1,2,17,0,0,Suburban,No,No,Excellent,5.6,351659
1,H00002,3334,6,2,1,0,3,0,Rural,No,Yes,Excellent,10.1,807321
2,H00003,1435,1,1,3,9,1,0,Suburban,No,No,Average,14.6,340628
3,H00004,3099,3,5,2,13,2,0,Suburban,No,No,Needs Renovation,11.9,675886
4,H00005,2086,6,1,2,16,1,2,Suburban,No,Yes,Needs Renovation,12.2,527347


In [39]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   House_ID             5000 non-null   object 
 1   Area_sqft            5000 non-null   int64  
 2   Bedrooms             5000 non-null   int64  
 3   Bathrooms            5000 non-null   int64  
 4   Floors               5000 non-null   int64  
 5   House_Age            5000 non-null   int64  
 6   Garage               5000 non-null   int64  
 7   Parking              5000 non-null   int64  
 8   Location             5000 non-null   object 
 9   Furnished            5000 non-null   object 
 10  Garden               5000 non-null   object 
 11  Condition            5000 non-null   object 
 12  Distance_to_City_km  5000 non-null   float64
 13  Price                5000 non-null   int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 547.0+ KB


In [40]:
X = df[['Area_sqft',
        'Bedrooms',
        'Bathrooms',
        'Floors',
        'House_Age',
        'Garage',
        'Parking',
        'Location',
        'Furnished',
        'Garden',
        'Condition',
        'Distance_to_City_km',]]

y = df['Price']

In [41]:
from sklearn.preprocessing import LabelEncoder

X = X.copy()

label_encoders = {}

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X.loc[:, col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 4000
Testing Samples: 1000


In [43]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42
)

model

RandomForestRegressor(random_state=42)

In [44]:
print(model)

RandomForestRegressor(random_state=42)


In [45]:
model.fit(X_train, y_train)
print("Model Training Completed.")


Model Training Completed.


In [46]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict
y_pred = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation")
print("-------------------------")
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R² Score:", r2)


Model Evaluation
-------------------------
Mean Absolute Error (MAE): 24371.772100000002
Mean Squared Error (MSE): 922231547.0567272
Root Mean Squared Error (RMSE): 30368.265460126746
R² Score: 0.9841335515043617


In [47]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

params = {
    "n_estimators": [20, 30, 50],
    "max_depth": [5, 10]
}

grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=params,
    cv=3,
    scoring="r2"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)
print("Best R² Score:", grid.best_score_)

Best Parameters: {'max_depth': 10, 'n_estimators': 50}
Best R² Score: 0.9816405845891527


In [48]:
import joblib

joblib.dump(
    best_model,
    "house_price_model.pkl",
    compress=9
)

print("Model Saved Successfully.")

Model Saved Successfully.


In [49]:
loaded_model = joblib.load(
    "house_price_model.pkl"
)

prediction = loaded_model.predict(X_test)

print("Sample Predictions:")
print(prediction)

Sample Predictions:
[ 416188.12502132  902292.36143956  729186.66598927  626122.29337872
  330548.54681987  724885.4644282   437886.03409674 1098032.66831914
  534664.53485159  471967.72385714  943927.25188391  870074.71296219
  854981.04182434  801259.5625111   349177.80756976  741373.64044208
  758498.75753386  392284.57277478  547765.31450467  702673.13108312
  351787.30122027  927497.79085462 1012492.32078804  848028.37292937
  868148.77634511  289807.35385498  674389.88434076  369068.08950147
  831329.17912142  418497.23594391  333467.92475874  476938.51335437
  484605.41383333  363964.86940508 1136579.17093889 1029296.13929437
  977136.98760458  483805.22510914  913653.34574176  780606.18474339
  934008.15162376  417354.69404266  326836.19243843  361093.34276886
  630898.51827675  391848.16676021  822436.37560729  501247.60602132
  335313.99408325  816613.88101557  661981.54982152  758369.95566414
  920735.26227103  711271.97580272  844135.09180857  583228.84295566
  838091.48234

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

prediction = best_model.predict(X_test)

mae = mean_absolute_error(y_test, prediction)
mse = mean_squared_error(y_test, prediction)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, prediction)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE : 25540.280832742563
MSE : 1016800731.334939
RMSE: 31887.313015287742
R² Score: 0.9825065446030975


In [51]:
print(X.columns.tolist())

['Area_sqft', 'Bedrooms', 'Bathrooms', 'Floors', 'House_Age', 'Garage', 'Parking', 'Location', 'Furnished', 'Garden', 'Condition', 'Distance_to_City_km']
